# Solved Example: Training a CartPole Agent with PPO

**Solved Examples Notebook -- Week 1**

This notebook shows you how to train an agent on **CartPole-v1** using Proximal Policy Optimization (PPO).  
CartPole is a simple balancing problem and a standard benchmark for RL. It is *not* the walking robot.  
The purpose is to get comfortable with the Stable-Baselines3 API before using it on a harder problem.

**You will learn:**
- How to use a Gymnasium environment
- How to create and train a PPO agent with SB3
- How to read the TensorBoard training metrics
- How to evaluate a trained agent
- What the key PPO hyperparameters do

---
**This notebook is fully solved. All cells will run end-to-end.  
Run them in order and read the explanations.**

---
## 0. Install / verify dependencies

Run the cell below once. If you set up the project with `pip install -r requirements.txt` the packages are already there.

In [ ]:
# Quick package check -- no install, just verify
import importlib

required = {
    'gymnasium':          'gymnasium',
    'stable_baselines3':  'stable-baselines3',
    'tensorboard':        'tensorboard',
    'matplotlib':         'matplotlib',
    'numpy':              'numpy',
}

all_ok = True
for module, pip_name in required.items():
    try:
        importlib.import_module(module)
        print(f'  OK  {pip_name}')
    except ImportError:
        print(f'  MISSING  {pip_name}  -- run: pip install {pip_name}')
        all_ok = False

if all_ok:
    print('\nAll dependencies are present.')

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import os

from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback

print('Imports OK.')
print('Gymnasium version:', gym.__version__)
import stable_baselines3
print('SB3 version:', stable_baselines3.__version__)

---
## 1. Understanding CartPole

CartPole-v1 is a classic control problem:
- A cart moves along a track.
- A pole is attached to the cart and can rotate freely.
- The agent must keep the pole upright by pushing the cart left or right.

### Observation space (4-dimensional)

| Index | Meaning | Typical range |
|-------|---------|---------------|
| 0 | Cart position | -4.8 to 4.8 |
| 1 | Cart velocity | -inf to inf |
| 2 | Pole angle (radians) | -0.418 to 0.418 |
| 3 | Pole angular velocity | -inf to inf |

### Action space (discrete)
0 = push left, 1 = push right

### Reward
+1 for every timestep the pole does not fall. Max episode length = 500 steps, so max score = 500.

In [ ]:
# Create a single environment and explore it
env = gym.make('CartPole-v1')

print('Observation space:', env.observation_space)
print('Action space:     ', env.action_space)

# Reset and take a few random steps
obs, info = env.reset(seed=0)
print('\nInitial observation:', obs)

for step_num in range(5):
    action = env.action_space.sample()       # random action
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    print(f'  step {step_num+1}: action={action}, obs={np.round(obs, 3)}, '
          f'reward={reward}, done={done}')

env.close()

---
## 2. Training with PPO

SB3 makes it very easy to train a PPO agent.  
The key components are:

- **`make_vec_env`**: Creates several copies of the environment running in parallel. This produces more training data per unit time.
- **`PPO(...)`**: Creates the agent. The policy `'MlpPolicy'` means the neural network input is the raw observation vector (as opposed to an image-based policy).
- **`model.learn(total_timesteps)`**: Runs the training loop.

### Key PPO parameters

| Parameter | What it controls | CartPole default |
|-----------|-----------------|------------------|
| `learning_rate` | Step size for gradient updates | 3e-4 |
| `n_steps` | Steps collected per env before each update | 2048 |
| `batch_size` | Mini-batch size used in the update | 64 |
| `n_epochs` | How many passes over the data each update | 10 |
| `gamma` | Discount factor | 0.99 |
| `gae_lambda` | GAE smoothing parameter | 0.95 |
| `clip_range` | Clip the policy ratio at this value | 0.2 |
| `ent_coef` | Entropy bonus (encourages exploration) | 0.0 |

In [ ]:
TENSORBOARD_LOG_DIR = './tb_logs_cartpole'
MODEL_SAVE_DIR      = './models_cartpole'
os.makedirs(TENSORBOARD_LOG_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# 4 parallel environments
vec_env = make_vec_env('CartPole-v1', n_envs=4, seed=0)

# Create the PPO agent
model = PPO(
    policy        = 'MlpPolicy',
    env           = vec_env,
    learning_rate = 3e-4,
    n_steps       = 2048,
    batch_size    = 64,
    n_epochs      = 10,
    gamma         = 0.99,
    gae_lambda    = 0.95,
    clip_range    = 0.2,
    ent_coef      = 0.0,
    verbose       = 1,
    tensorboard_log = TENSORBOARD_LOG_DIR,
)

print('Model created.  Network architecture:')
print(model.policy)

In [ ]:
# EvalCallback: evaluates the agent on a separate environment every 5000 steps
# and saves the best model seen so far.
eval_env = make_vec_env('CartPole-v1', n_envs=1, seed=99)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path = MODEL_SAVE_DIR,
    log_path             = MODEL_SAVE_DIR,
    eval_freq            = 5000,
    n_eval_episodes      = 20,
    deterministic        = True,
    verbose              = 0,
)

# Train for 100k steps (should reach near-perfect score on CartPole)
print('Training...  (this takes about 30-60 seconds)')
model.learn(
    total_timesteps = 100_000,
    callback        = eval_callback,
    tb_log_name     = 'ppo_cartpole',
    reset_num_timesteps = True,
)

# Save the final model
model.save(os.path.join(MODEL_SAVE_DIR, 'final_model'))
print('\nTraining done.  Model saved.')

---
## 3. Evaluating the Trained Agent

In [ ]:
# Load the best model (saved by EvalCallback)
best_model_path = os.path.join(MODEL_SAVE_DIR, 'best_model')
best_model = PPO.load(best_model_path)

# Evaluate on 50 episodes
eval_env_single = gym.make('CartPole-v1')
mean_reward, std_reward = evaluate_policy(
    best_model, eval_env_single,
    n_eval_episodes = 50,
    deterministic   = True,
)
eval_env_single.close()

print(f'Mean reward over 50 episodes: {mean_reward:.1f} +/- {std_reward:.1f}')
print(f'(Max possible on CartPole-v1 = 500)')

if mean_reward >= 490:
    print('Excellent -- the agent has essentially solved CartPole!')
elif mean_reward >= 400:
    print('Good -- the agent is doing well but not perfect.')
else:
    print('Needs more training -- try increasing total_timesteps.')

---
## 4. Reading the Evaluation Log

The `EvalCallback` saved evaluation results in `models_cartpole/evaluations.npz`.  
Let us load and plot them to see how performance improved over training.

In [ ]:
eval_data = np.load(os.path.join(MODEL_SAVE_DIR, 'evaluations.npz'))

timesteps    = eval_data['timesteps']          # x-axis: training steps
ep_rewards   = eval_data['results']            # shape (n_evals, n_eval_episodes)
mean_rewards = ep_rewards.mean(axis=1)
std_rewards  = ep_rewards.std(axis=1)

plt.figure(figsize=(9, 4))
plt.fill_between(timesteps,
                 mean_rewards - std_rewards,
                 mean_rewards + std_rewards,
                 alpha=0.3, color='steelblue', label='Mean +/- 1 std')
plt.plot(timesteps, mean_rewards, color='steelblue', linewidth=2, label='Mean reward')
plt.axhline(500, color='green', linestyle='--', linewidth=1, label='Max score (500)')
plt.xlabel('Training timesteps')
plt.ylabel('Mean episode reward')
plt.title('PPO on CartPole-v1: Evaluation Curve')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

---
## 5. Using TensorBoard

TensorBoard gives a richer view of training. To launch it:

```bash
tensorboard --logdir ./tb_logs_cartpole
```

Then open `http://localhost:6006` in a browser.

### Metrics to look at

| Metric | What it means | Healthy sign |
|--------|--------------|-------------|
| `rollout/ep_rew_mean` | Average episode reward over recent rollout | Increasing |
| `rollout/ep_len_mean` | Average episode length | Increasing (for CartPole) |
| `train/value_loss` | How wrong the value function is | Decreasing |
| `train/policy_gradient_loss` | Policy update magnitude | Small and stable |
| `train/entropy_loss` | Policy entropy (how random actions are) | Decreasing slowly |
| `train/approx_kl` | How much the policy changed per update | Should stay below 0.1 |
| `train/clip_fraction` | Fraction of updates that hit the clip | Should stay below 0.2 |
| `train/explained_variance` | How well value function predicts returns | Should approach 1.0 |

---
## 6. What Happens if We Change Hyperparameters?

Let us do a quick comparison: train with a bad learning rate vs the default.

In [ ]:
def train_and_eval(lr, run_name, total_steps=60_000, n_envs=4, seed=0):
    """Train PPO on CartPole and return evaluation results."""
    venv = make_vec_env('CartPole-v1', n_envs=n_envs, seed=seed)
    save_dir = f'./models_cartpole/{run_name}'
    os.makedirs(save_dir, exist_ok=True)

    m = PPO('MlpPolicy', venv, learning_rate=lr, verbose=0,
            tensorboard_log=TENSORBOARD_LOG_DIR)

    ev_env  = make_vec_env('CartPole-v1', n_envs=1, seed=99)
    cb = EvalCallback(ev_env, best_model_save_path=save_dir,
                      log_path=save_dir, eval_freq=5000,
                      n_eval_episodes=20, deterministic=True, verbose=0)

    m.learn(total_timesteps=total_steps, callback=cb,
            tb_log_name=run_name, reset_num_timesteps=True)
    ev_env.close()
    venv.close()

    data = np.load(os.path.join(save_dir, 'evaluations.npz'))
    return data['timesteps'], data['results'].mean(axis=1)


print('Training with default LR=3e-4 ...')
ts_good, rew_good = train_and_eval(lr=3e-4, run_name='lr_good')

print('Training with bad LR=0.05 (too large) ...')
ts_bad, rew_bad   = train_and_eval(lr=5e-2, run_name='lr_bad')

print('Training with small LR=1e-5 (too small) ...')
ts_slow, rew_slow = train_and_eval(lr=1e-5, run_name='lr_slow')

print('Done.')

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(ts_good,  rew_good,  label='LR = 3e-4 (default)',    linewidth=2, color='steelblue')
plt.plot(ts_bad,   rew_bad,   label='LR = 0.05 (too large)',  linewidth=2, color='tomato')
plt.plot(ts_slow,  rew_slow,  label='LR = 1e-5 (too small)',  linewidth=2, color='orange')
plt.axhline(500, color='green', linestyle='--', linewidth=1, label='Max score')
plt.xlabel('Training timesteps')
plt.ylabel('Mean episode reward')
plt.title('Effect of Learning Rate on PPO Training (CartPole-v1)')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

**What you should observe:**

- **LR = 3e-4 (default):** Learns steadily and reaches near-perfect score.
- **LR = 0.05 (too large):** Updates are too large and the policy becomes unstable. Performance may oscillate or diverge.
- **LR = 1e-5 (too small):** Updates are too small. Learning is correct but very slow.

This exact pattern appears in the Walker robot project too. In Week 3 you will run a systematic ablation study across multiple hyperparameters.

---
## 7. Connection to the Walker Robot Project

CartPole and the Walker2D robot are both continuous-state RL problems solved by PPO.  
Here is a comparison:

| Feature | CartPole-v1 | Walker2D (our project) |
|---------|------------|------------------------|
| Observation dim | 4 | 22 |
| Action type | Discrete (2 actions) | Continuous (6 joint torques) |
| Reward | +1 per step alive | Custom (velocity + energy + ...) |
| Episode length | up to 500 steps | up to 1000 steps |
| Train time | 1-2 minutes | 1-2 hours |
| SB3 policy | `MlpPolicy` | `MlpPolicy` |

The SB3 code you wrote here (create env, create PPO, learn, evaluate, callback) is **exactly** the same pattern used in `train.py` in this project. The only difference is that `train.py` wraps a PyBullet physics simulation instead of CartPole.

---
## 8. Reflection Questions

**Q1.** What does `n_envs=4` do? Why is running multiple environments at once helpful?

**Q2.** The `EvalCallback` uses `deterministic=True`. What does that mean for a PPO policy?

**Q3.** You saw that `explained_variance` should approach 1.0.  
What does `explained_variance = 0` mean? What does it mean for training?

**Q4.** In the learning rate comparison, why does a too-large LR cause instability?  
(Think about what happens to the gradient step size.)

**Q5.** CartPole has a discrete action space. The Walker2D uses continuous actions.  
SB3 PPO handles both with `MlpPolicy`. How is the output layer different in the two cases?